# HR Analytics: Employee Experience, Performance & Training

**Project type:** Exploratory Data Analysis (EDA)  
**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn  

This notebook explores employee status, performance ratings, employee survey scores, and training outcomes/costs. The source CSV is shared in anonymized form in `data/`.

## 1. Business questions
- What is the distribution of active and terminated employee records?
- How are performance scores distributed?
- Which departments contribute the most records?
- What do engagement, satisfaction and work-life balance scores look like?
- How are training outcomes and costs distributed?

**Scope note:** This dataset does not contain exit reasons or a labelled attrition target suitable for causal explanation or prediction. The project therefore reports descriptive HR insights, not an attrition prediction model.

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nsns.set_theme(style='whitegrid')\nDATA_PATH = 'data/hr_analytics_anonymized.csv'\ndf = pd.read_csv(DATA_PATH)\ndf.head()

## 2. Data overview and quality checks

In [ ]:
print('Shape:', df.shape)\ndisplay(df.info())\ndisplay(df.describe(include='all').T)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())\nmissing = df.isna().sum().sort_values(ascending=False)\ndisplay(missing[missing > 0])\nprint('Unique record IDs:', df['Record ID'].nunique())

## 3. Employee status and performance

In [ ]:
status_counts = df['EmployeeStatus'].value_counts()\nax = status_counts.plot(kind='bar', title='Employee Status Distribution', ylabel='Records', xlabel='Status')\nplt.xticks(rotation=0); plt.tight_layout(); plt.show()\nprint((status_counts / len(df) * 100).round(1))

In [ ]:
perf_counts = df['Performance Score'].value_counts()\nperf_counts.plot(kind='bar', title='Performance Score Distribution', ylabel='Records', xlabel='Performance score')\nplt.xticks(rotation=15); plt.tight_layout(); plt.show()\nprint((perf_counts / len(df) * 100).round(1))

## 4. Department-level view

In [ ]:
dept_counts = df['DepartmentType'].value_counts()\ndept_counts.plot(kind='bar', title='Records by Department', ylabel='Records', xlabel='Department')\nplt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()\ndisplay(df.groupby('DepartmentType').agg(\n    records=('Record ID','count'),\n    avg_rating=('Current Employee Rating','mean'),\n    avg_engagement=('Engagement Score','mean'),\n    avg_satisfaction=('Satisfaction Score','mean'),\n    avg_work_life_balance=('Work-Life Balance Score','mean'),\n    avg_training_cost=('Training Cost','mean')\n).round(2).sort_values('records', ascending=False))

## 5. Employee survey scores

In [ ]:
survey_cols = ['Engagement Score','Satisfaction Score','Work-Life Balance Score']\nscore_means = df[survey_cols].mean().sort_values(ascending=False)\nscore_means.plot(kind='bar', title='Average Employee Survey Scores', ylabel='Average score (1–5)', xlabel='Measure')\nplt.xticks(rotation=10); plt.ylim(0,5); plt.tight_layout(); plt.show()\ndisplay(df[survey_cols].describe().round(2))

## 6. Training outcomes and investment

In [ ]:
outcome_counts = df['Training Outcome'].value_counts()\noutcome_counts.plot(kind='bar', title='Training Outcome Distribution', ylabel='Records', xlabel='Outcome')\nplt.xticks(rotation=0); plt.tight_layout(); plt.show()\nprint((outcome_counts / len(df) * 100).round(1))

In [ ]:
training_summary = df.groupby('Training Outcome').agg(\n    records=('Record ID','count'),\n    avg_duration_days=('Training Duration(Days)','mean'),\n    avg_training_cost=('Training Cost','mean'),\n    total_training_cost=('Training Cost','sum')\n).round(2)\ndisplay(training_summary.sort_values('records', ascending=False))

In [ ]:
cost_by_outcome = df.groupby('Training Outcome')['Training Cost'].mean().sort_values(ascending=False)\ncost_by_outcome.plot(kind='bar', title='Average Training Cost by Outcome', ylabel='Average cost', xlabel='Training outcome')\nplt.xticks(rotation=15); plt.tight_layout(); plt.show()\nprint('Total recorded training cost:', round(df['Training Cost'].sum(), 2))

## 7. Correlation check

In [ ]:
numeric_cols = ['Current Employee Rating','Engagement Score','Satisfaction Score',\n                'Work-Life Balance Score','Training Duration(Days)','Training Cost','Age']\ncorr = df[numeric_cols].corr(numeric_only=True)\nplt.figure(figsize=(8,6))\nsns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0)\nplt.title('Correlation Matrix of Numeric HR Measures')\nplt.tight_layout(); plt.show()

## 8. Key findings (descriptive)
- The dataset contains **2,845 records and 28 source columns** before anonymization.
- Employee status: **2,458 Active (86.4%)** and **387 Terminated (13.6%)**.
- **Fully Meets** is the most common performance category (**2,251 records; 79.1%**).
- Production is the largest department group (**1,910 records**).
- Average scores: engagement **2.94/5**, satisfaction **3.03/5**, work-life balance **2.99/5**.
- Training outcomes are spread across Completed (**737**), Incomplete (**731**), Passed (**709**) and Failed (**668**).
- Total recorded training cost is **1,591,148.63** (currency/unit not specified in the source, so no currency symbol is assumed).

These are descriptive summaries of the provided dataset. They do not establish why employees leave or whether training causes changes in performance.

## 9. Recommendations and limitations
**Recommended follow-up:** review training completion barriers; compare survey scores by department; validate the meaning and scale of employee ratings with the data owner; and obtain exit reasons/tenure history before conducting attrition root-cause analysis.

**Limitations:** source data definitions and currency are not supplied; survey scales are assumed to be 1–5 based on observed values; the dataset contains employee-related fields and has therefore been anonymized for public sharing. Results should be interpreted as an exploratory portfolio analysis, not an audited HR report.

## 10. Reproducibility
Run the notebook from the project root so the relative CSV path resolves. Install dependencies with `pip install -r requirements.txt`.